# Exp 11 - Secure V2X Communication using Encryption and SHA-256 in Python

> Scope note: this notebook is a lab-scale deterministic simulation. It is intended for understanding timing, communication, and security concepts. It is not a certification model for a production autonomous vehicle.

## Objective

Demonstrate message confidentiality and integrity checks for V2X-style messages.

The notebook uses SHA-256 through HMAC for integrity/authentication demonstration. It also uses a simple XOR transform only to illustrate reversible encoding. XOR is not production-grade encryption.

## Core Real-Time Systems Theory Notes

### 1. Introduction to Real-Time Systems

A real-time system is a computing system in which correctness depends on two things: the logical correctness of the output and the time at which the output is produced. In a normal general-purpose system, a late answer may be inconvenient. In a real-time system, a late answer may be useless or may cause unsafe behavior.

Logical correctness means that the calculated value or decision is correct. Temporal correctness means that the value or decision is available within the required time bound. A vehicle braking controller, robotic arm controller, industrial motor drive, medical monitoring device, avionics controller, or power-grid protection unit must satisfy both.

Real-time does not simply mean "fast." A fast system that sometimes misses its deadline is not dependable for hard real-time control. A slower system with bounded and predictable timing may be more suitable if it always meets the required deadline. The key engineering properties are determinism, predictability, bounded latency, and analyzable worst-case behavior.

General-purpose systems optimize average response, throughput, fairness, and user convenience. Real-time systems optimize deadline satisfaction, bounded response time, and predictable behavior under defined load. This is why real-time operating systems, embedded controllers, field buses, and deterministic networks often use priority policies, static configuration, time slots, or admission control.

### 2. Classification of Real-Time Systems

Hard real-time systems must not miss deadlines. A missed deadline is treated as system failure. Examples include autonomous emergency braking, airbag control, flight-control surfaces, pacemaker control, and industrial safety shutdown.

Firm real-time systems can tolerate some missed deadlines, but a late result has no value and is discarded. Examples include object-detection frames that arrive after the object is no longer relevant, traffic-sign recognition after the vehicle has passed the sign, or a stale cooperative-awareness message in V2X communication.

Soft real-time systems tolerate deadline misses with quality degradation. Examples include dashboard display refresh, infotainment audio buffering, non-critical telemetry upload, passenger comfort control, and route-estimation updates.

The classification depends on the consequence of lateness, not only the application name. A camera pipeline may be hard real-time when used for emergency braking, firm real-time when used for immediate object tracking, and soft real-time when used for driver display recording.

### 3. Real-Time Tasks and Events

A task is a schedulable unit of computation. In autonomous systems, tasks may represent sensor sampling, frame processing, message transmission, controller update, actuator command generation, logging, or security checking.

Periodic tasks occur at fixed intervals. Example: sample wheel speed every 10 ms. A periodic task is commonly described by execution time C, period T, and deadline D.

Aperiodic tasks occur irregularly and do not have a guaranteed minimum inter-arrival time. Example: a user opens a diagnostic screen. Aperiodic work is often lower criticality or handled by background servers.

Sporadic tasks occur irregularly but have a known minimum separation between arrivals. Example: emergency obstacle events may occur unpredictably but cannot arrive faster than a defined physical or system limit. Sporadic modelling is useful because it allows worst-case analysis.

Time-triggered events are released by a clock schedule. They improve predictability because activation times are known in advance. Event-triggered events are released when an external condition occurs, such as receiving a packet, detecting an obstacle, or crossing a threshold. Event-triggered systems are responsive but require careful overload handling.

### 4. Timing Parameters

The event occurrence time is the real-world time at which the physical event occurs. Release time is when the corresponding task becomes ready for scheduling. Arrival time is often used for the time at which a job enters a queue or a packet reaches a node. Start time is when execution actually begins. Execution time or computation time is the CPU or processor time consumed by the job.

Waiting time is the time spent ready but not executing:

```
waiting_time = start_time - release_time
```

Completion time or finish time is when the job finishes. Response time is the delay from release or arrival to completion:

```
response_time = finish_time - release_time
```

In many lab contexts, turnaround time is also:

```
turnaround_time = finish_time - arrival_time
```

If release time and arrival time are the same, response time and turnaround time become numerically equal. In networked systems they may differ because a real-world event can occur before the software task is released, or a packet can be generated before it reaches the receiving queue.

### 5. Timing Constraints

A relative deadline is measured from release time. An absolute deadline is a time on the system timeline:

```
absolute_deadline = release_time + relative_deadline
```

A deadline is met when:

```
finish_time <= absolute_deadline
```

A deadline miss occurs when:

```
finish_time > absolute_deadline
```

Deadline margin shows how much time remains at completion:

```
deadline_margin = absolute_deadline - finish_time
```

Positive margin means the task finished early. Zero means it finished exactly at the deadline. Negative margin means a miss.

Slack time estimates available spare time before a deadline:

```
slack = absolute_deadline - current_time - remaining_execution_time
```

Laxity is often used similarly:

```
laxity = deadline - current_time - remaining_computation_time
```

Worst-Case Execution Time, or WCET, is the maximum execution time under defined assumptions. Best-Case Execution Time, or BCET, is the minimum. Average execution time is not enough for hard real-time certification because rare long execution paths still matter.

### 6. Communication Performance Parameters

Latency is the time taken for data to move from source to destination:

```
latency = receive_time - send_time
```

Jitter is variation in latency. A simple packet-to-packet jitter estimate is:

```
jitter_i = abs(latency_i - latency_(i-1))
```

Throughput is useful delivered data per unit time:

```
throughput = delivered_bits / observation_time
```

Bandwidth is the nominal or available capacity of a link. Throughput is what is actually achieved after overhead, contention, retransmission, protocol limits, and congestion.

Packet transmission time is:

```
transmission_time = packet_size_bits / link_rate_bits_per_second
```

End-to-end delay can be modeled as:

```
end_to_end_delay = processing_delay + queueing_delay + transmission_delay + propagation_delay
```

Communication overhead is the extra data or time consumed by headers, acknowledgements, encryption, retransmission, routing, and synchronization. Packet loss affects reliability:

```
packet_loss_rate = lost_packets / sent_packets
reliability = delivered_packets / sent_packets
```

### 7. Real-Time Communication Requirements

Bounded latency means there is a known upper limit for message delay under defined conditions. Low jitter means delay stays stable across transmissions. Predictable communication means the designer can reason about message timing before deployment. Reliability means messages are delivered with acceptable probability or with recovery mechanisms. Availability means the communication service is usable when needed.

Deterministic message delivery is often achieved through priority arbitration, time slots, traffic shaping, redundancy, admission control, or real-time Ethernet features. Deadline-aware communication means messages are scheduled according to urgency and usefulness, not simply first-come first-served.

### 8. Timing Analysis in Autonomous Systems

A typical autonomous timing chain is:

```
Sensor -> Perception -> Planning/Control -> Actuator -> Physical Response
```

The perception-to-action delay is:

```
perception_to_action_delay =
    sensor_capture_time
  + sensor_preprocessing_time
  + perception_inference_time
  + planning_time
  + control_time
  + communication_time
  + actuator_response_time
```

For an autonomous braking example:

```
stopping_distance = reaction_distance + braking_distance
reaction_distance = vehicle_speed * total_system_delay
braking_distance = vehicle_speed^2 / (2 * deceleration)
```

Deadline verification compares the computed or measured response time with the maximum safe response time:

```
system_is_timely = measured_response_time <= required_deadline
```

Case Study - Autonomous Emergency Braking:
A front sensor detects an obstacle at a fixed distance. The system must capture sensor data, process it, decide, transmit the command, and apply braking before the remaining stopping distance becomes unsafe. The case study shows why real-time correctness is a chain property. A fast perception algorithm alone is not enough if the actuator command is delayed.

Case Study - Robotic Arm in Industrial Automation:
A robotic arm must stop when a worker crosses a safety boundary. Sensor detection, controller scheduling, network delivery, and motor-drive response must all be bounded. High average throughput is irrelevant if one delayed safety packet allows the arm to continue moving too long.

Case Study - V2X Hazard Warning:
A vehicle broadcasts a hazard message to nearby vehicles. The message is useful only if received before the receiving vehicle must react. This connects communication latency, jitter, packet loss, message freshness, and security verification.

### 9. Textbook Design Workflow for Real-Time Experiments

When solving a real-time lab problem, use a disciplined workflow. First identify the physical event or communication event. Second identify the software task or network message created by that event. Third list the timing parameters: release time, start time, execution time, finish time, and deadline. Fourth compute the response time and deadline margin. Fifth classify the consequence of lateness as hard, firm, or soft. Sixth propose a design improvement if the deadline is missed.

For autonomous systems, the timing boundary should be tied to a physical reason. For example, a braking deadline should relate to speed, distance, and deceleration. A communication deadline should relate to how long a message remains useful. A security verification deadline should relate to whether authentication or IDS checks finish before the receiver uses the message.

### 10. Common Architectures Used Across These Experiments

Most experiments in this lab can be understood using one of three architecture patterns.

Control-loop pattern:

```
Sensor -> Controller Task -> Actuator -> Plant / Vehicle -> Sensor
```

Communication-loop pattern:

```
Publisher / Sender -> Network Medium -> Receiver / Subscriber -> Application Decision
```

Security-monitoring pattern:

```
Message Source -> Security Check -> IDS / Risk Logic -> Accept, Reject, or Alert
```

The control-loop pattern focuses on WCET, response time, and deadline satisfaction. The communication-loop pattern focuses on latency, jitter, throughput, packet loss, and deterministic delivery. The security-monitoring pattern focuses on integrity, authentication, replay resistance, anomaly detection, and risk reduction. Autonomous systems usually combine all three patterns, which is why timing and security cannot be treated as separate afterthoughts.

### 11. Common Mistakes to Avoid in Lab Answers

Do not say "real-time means fast." Say "real-time means deadline-bound." Do not use average execution time as a substitute for WCET in hard real-time analysis. Do not conclude that high throughput guarantees good real-time performance. Do not claim a security mechanism provides authentication unless the mechanism actually proves sender identity. Do not claim a physical simulator or broker was used if the notebook uses a Python fallback. Clear assumptions make the lab record more credible.

### Core References for These Notes

- Python timing functions such as `perf_counter()` and monotonic clocks are documented by the official Python `time` module documentation: https://docs.python.org/3/library/time.html
- IEEE 802.1 Time-Sensitive Networking is the IEEE working-group area for time-sensitive network behavior: https://1.ieee802.org/tsn/
- SUMO official documentation describes traffic simulation concepts used in V2V mobility experiments: https://sumo.dlr.de/docs/
- MQTT is an OASIS publish-subscribe messaging standard for IoT telemetry: https://docs.oasis-open.org/mqtt/mqtt/v5.0/mqtt-v5.0.html
- NIST FIPS 180-4 specifies SHA-256 as part of the Secure Hash Standard: https://csrc.nist.gov/pubs/fips/180-4/upd1/final
- NIST SP 800-30 Rev. 1 provides risk-assessment guidance: https://csrc.nist.gov/pubs/sp/800/30/r1/final

## Extended Experiment Notes and Case Studies

### Experiment Focus

Secure V2X communication protects messages from unauthorized reading, modification, impersonation, and reuse. This experiment demonstrates encryption-like transformation and SHA-256 hashing for integrity checking. It is educational and should not be described as a complete production V2X security stack.

### Experiment Architecture

```
Sender Message
  -> Encode / Encrypt
  -> Hash or Integrity Value
  -> Transmit
  -> Receiver Verification
  -> Accept or Reject
```

Confidentiality, integrity, authentication, and freshness are separate security goals. A hash alone is not authentication unless it is bound to a secret or signature scheme.

### Detailed Formula Set

```
digest = SHA256(message)
integrity_valid = received_digest == SHA256(received_message)
freshness_valid = timestamp_or_nonce_is_new
```

### Case Study 1 - Tampered Hazard Warning

An attacker modifies a hazard warning payload. A recomputed digest will differ from the trusted digest, so the receiver can detect tampering.

### Case Study 2 - Emergency Vehicle Claim

A message claiming emergency priority must be authenticated. Encryption alone does not prove sender authorization.

### Case Study 3 - Location Privacy

Vehicle location telemetry may need confidentiality and minimization. Even encrypted systems can leak metadata if topic names or transmission patterns reveal behavior.

### Additional Security Discussion

For V2X, message protection must happen before the receiver acts on the message. This creates a timing-security tradeoff. Strong verification is necessary, but verification that finishes too late can still fail the real-time requirement. A practical design therefore budgets time for parsing, certificate or key checks, hash or signature verification, freshness checking, and application decision-making.

Key management is a major part of secure communication. If two vehicles do not share the correct trust relationship, the receiver cannot safely decide whether a message is genuine. In a lab notebook, it is acceptable to demonstrate hashing with a small message, but the conclusion should state that production V2X normally requires authenticated integrity and a trust infrastructure.

Another important point is freshness. A message can be encrypted and integrity-protected but still dangerous if an old valid message is replayed. Therefore timestamps, nonces, sequence numbers, and replay caches are security controls that directly support real-time safety.

### Lab Record Guidance

Separate the security properties: confidentiality, integrity, authentication, freshness, and availability. State exactly which property the notebook demonstrates.

## Architecture

```text
V2X Message
  |-- vehicle id
  |-- event
  |-- position
  |-- timestamp
          |
          v
Sender Security Layer
  |-- reversible encoding demo
  |-- HMAC-SHA256 tag
          |
          v
Receiver Security Layer
  |-- decode
  |-- recompute HMAC
  |-- compare tags
          |
          v
Accept / Reject
```

## Formulas and Required Theory

HMAC concept:

\[
tag = HMAC_{SHA256}(key, message)
\]

Integrity verification:

\[
\text{valid} = compare\_digest(tag_{received}, HMAC_{SHA256}(key, message_{received}))
\]

If a message changes, the recomputed tag changes. A receiver rejects messages whose tags do not match.

## In-Lab Method

1. Define a byte message and shared key.
2. Generate an HMAC-SHA256 tag.
3. Encode and decode the message.
4. Recompute the tag at the receiver.
5. Use constant-time comparison for verification.

In [1]:
import hashlib
import hmac

print("EXP 11 - IN-LAB SECURE V2X MESSAGE")
key = b"lab-shared-key"
message = b"vehicle=V1;event=hard_brake;x=125;y=48;ts=1002"
digest = hmac.new(key, message, hashlib.sha256).hexdigest()
cipher = bytes(b ^ key[i % len(key)] for i, b in enumerate(message))
plain = bytes(b ^ key[i % len(key)] for i, b in enumerate(cipher))
print("Ciphertext hex:", cipher.hex())
print("SHA-256 HMAC :", digest)
print("Decrypted    :", plain.decode())
print("Integrity ok :", hmac.compare_digest(digest, hmac.new(key, plain, hashlib.sha256).hexdigest()))

EXP 11 - IN-LAB SECURE V2X MESSAGE
Ciphertext hex: 1a040a441004044f3355160e131c02155f45121a052d07164c000042145c531f4653184f515c161f16445d51521f
SHA-256 HMAC : d0e7612d4b2fe8a949d2e3ef0ea38c5db5b0170718412c315b29907b1f09331e
Decrypted    : vehicle=V1;event=hard_brake;x=125;y=48;ts=1002
Integrity ok : True


## Post-Lab Method

The post-lab cell changes the speed field from the original message. The HMAC check fails for the tampered message, proving that modification is detected.

In [2]:
import hashlib
import hmac

print("EXP 11 - POST-LAB TAMPER CHECK")
key = b"lab-shared-key"
original = b"vehicle=V1;speed=45;ts=1002"
tampered = b"vehicle=V1;speed=95;ts=1002"
tag = hmac.new(key, original, hashlib.sha256).hexdigest()
print("Original valid:", hmac.compare_digest(tag, hmac.new(key, original, hashlib.sha256).hexdigest()))
print("Tampered valid:", hmac.compare_digest(tag, hmac.new(key, tampered, hashlib.sha256).hexdigest()))
print("Note: XOR encryption above is only a lab demonstration; production V2X needs vetted cryptography.")

EXP 11 - POST-LAB TAMPER CHECK
Original valid: True
Tampered valid: False
Note: XOR encryption above is only a lab demonstration; production V2X needs vetted cryptography.


## What to Write in the Lab Record

- Include ciphertext hex and HMAC output.
- Explain why hashing alone is not the same as authenticated integrity.
- Explain why `compare_digest` is used.
- Explicitly state that XOR is only a classroom demonstration, not production encryption.

## References

- Python `hmac` documentation: https://docs.python.org/3/library/hmac.html
- Python `hashlib` documentation: https://docs.python.org/3/library/hashlib.html